# 🅱️ Análisis Exploratorio de Datos (EDA) Categórico y Visualización
### 1. Frecuencias Absolutas, Relativas y Tablas de Contingencia

In [1]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

df_clean = pd.read_csv("../data/processed/telco_customer_churn_processed.csv")

# 1. Cálculo de Frecuencias Absolutas y Relativas
variables_categoricas = ['Churn', 'Contract', 'PaymentMethod']

print("Frecuencias Absolutas y relativas")
for col in variables_categoricas:
    freq_abs = df_clean[col].value_counts()
    freq_rel = (df_clean[col].value_counts(normalize=True) * 100).round(2)
    
    tabla_freq = pd.DataFrame({
        'Frecuencia Absoluta (Clientes)': freq_abs,
        'Frecuencia Relativa (%)': freq_rel
    })
    print(f"\n Distribución de la variable: {col}")
    display(tabla_freq)

# 2. Tablas de Contingencia respecto a la tasa de cancelación (Churn)
print("Tablas de contigencia")
variables_cruce = ['Contract', 'PaymentMethod', 'InternetService', 'SeniorCitizen']

for col in variables_cruce:
    # Conteo absoluto con totales
    ct_abs = pd.crosstab(df_clean[col], df_clean['Churn'], margins=True, margins_name='Total')
    # Porcentaje por fila (muestra la tasa de abandono de cada categoría)
    ct_rel = (pd.crosstab(df_clean[col], df_clean['Churn'], normalize='index') * 100).round(2)
    ct_rel.columns = ['Retenidos - No (%)', 'Tasa de Churn - Yes (%)']
    
    tabla_contingencia = pd.concat([ct_abs, ct_rel], axis=1)
    print(f"\n Cruce de {col} vs Churn:")
    display(tabla_contingencia)

Frecuencias Absolutas y relativas

 Distribución de la variable: Churn


,Frecuencia Absoluta (Clientes),Frecuencia Relativa (%)
Churn,,
No,5174,73.46
Yes,1869,26.54



 Distribución de la variable: Contract


,Frecuencia Absoluta (Clientes),Frecuencia Relativa (%)
Contract,,
Month-to-month,3875,55.02
Two year,1695,24.07
One year,1473,20.91



 Distribución de la variable: PaymentMethod


,Frecuencia Absoluta (Clientes),Frecuencia Relativa (%)
PaymentMethod,,
Electronic check,2365,33.58
Mailed check,1612,22.89
Bank transfer (automatic),1544,21.92
Credit card (automatic),1522,21.61


... (4 outputs left out)


 Cruce de InternetService vs Churn:


,No,Yes,Total,Retenidos - No (%),Tasa de Churn - Yes (%)
InternetService,,,,,
DSL,1962,459,2421,81.04,18.96
Fiber optic,1799,1297,3096,58.11,41.89
No,1413,113,1526,92.60,7.40
Total,5174,1869,7043,NaN,NaN



 Cruce de SeniorCitizen vs Churn:


,No,Yes,Total,Retenidos - No (%),Tasa de Churn - Yes (%)
SeniorCitizen,,,,,
0,4508,1393,5901,76.39,23.61
1,666,476,1142,58.32,41.68
Total,5174,1869,7043,NaN,NaN


### 2. Gráficos Comparativos de Cancelación (Churn)

In [ ]:
# Configuración estética de Seaborn
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico 1: Tasa de cancelación según Tipo de Contrato
ax1 = sns.countplot(
    data=df_clean,
    x='Contract',
    hue='Churn',
    palette=['#2ecc71', '#e74c3c'],
    order=['Month-to-month', 'One year', 'Two year'],
    ax=axes[0]
)
axes[0].set_title('Cancelación (Churn) según Tipo de Contrato', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Tipo de Contrato', fontsize=11)
axes[0].set_ylabel('Cantidad de Clientes', fontsize=11)
axes[0].legend(title='¿Canceló?', labels=['No (Continúa)', 'Sí (Churn)'])

for container in ax1.containers:
    ax1.bar_label(container, fontsize=10)

# Gráfico 2: Tasa de cancelación según Modalidad de Pago
ax2 = sns.countplot(
    data=df_clean,
    x='PaymentMethod',
    hue='Churn',
    palette=['#2ecc71', '#e74c3c'],
    ax=axes[1]
)
axes[1].set_title('Cancelación (Churn) según Modalidad de Pago', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Modalidad de Pago', fontsize=11)
axes[1].set_ylabel('Cantidad de Clientes', fontsize=11)
axes[1].tick_params(axis='x', rotation=15)
axes[1].legend(title='¿Canceló?', labels=['No (Continúa)', 'Sí (Churn)'])

for container in ax2.containers:
    ax2.bar_label(container, fontsize=10)

plt.tight_layout()
plt.show()

#### 💡 Interpretación Analítica Orientada al Impacto en el Negocio

1. **Análisis por Tipo de Contrato (`Contract`):**
   * **Hallazgo Estadístico:** Aunque la tasa general de cancelación de la empresa es del **26.54%** (1.869 bajas sobre 7.043 clientes), el abandono está fuertemente concentrado en los contratos mensuales (`Month-to-month`), los cuales representan el **55.02%** de la cartera total y sufren una tasa de cancelación crítica del **42.71%** (1.655 bajas). Por el contrario, en los contratos de 1 año (`One year`) la baja cae al **11.27%**, y en los de 2 años (`Two year`) es de apenas el **2.83%**.
   * **Impacto en el Negocio:** La falta de compromiso contractual a largo plazo es el principal impulsor de pérdida de ingresos recurrentes. Se recomienda diseñar incentivos comerciales (descuentos en el abono mensual o meses bonificados) para migrar a los clientes de modalidad mes a mes hacia contratos anuales.

2. **Análisis por Modalidad de Pago (`PaymentMethod`):**
   * **Hallazgo Estadístico:** Los clientes que abonan mediante cheque electrónico (`Electronic check`, el **33.58%** del total) registran una tasa de abandono sumamente elevada del **45.29%** (1.071 bajas). En contraste, quienes utilizan medios de pago automáticos presentan tasas de fuga casi tres veces menores: **16.71%** en transferencia bancaria automática y **15.24%** en tarjeta de crédito automática.
   * **Impacto en el Negocio:** Los métodos de pago manuales generan fricción mes a mes y exponen al cliente a reconsiderar la continuidad del servicio en cada vencimiento. La estrategia comercial debe enfocarse en bonificar la adhesión al débito automático con tarjeta o cuenta bancaria para fidelizar la cartera.